# Setup section

# Install the dataretrieval package
To download daily mean streamflow data measured at Lowman.

In [1]:
# Install the USGS package dataretrieval to get daily discharge time series
!pip install dataretrieval -q

In [2]:
import glob
import os
from dataretrieval import waterdata

from google.colab import auth
from google.auth import default
from google.colab import drive

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

from scipy.stats import linregress

In [3]:
# Running this cell will bring multiple prompts to access your drive. Accept and continue

# connect to your google drive (needed for saving files at the end)
drive.mount('/content/drive')
# Authenticate with Google Colab - do i need this for csv?
auth.authenticate_user()
creds, _ = default()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Edit the variables in this cell to select different parameter and save location.

In [4]:
# path to save figures
fig_path = '/content/drive/Shareddrives/RAPID_Wapiti_project/Rating Curves/figures/TSS'
variable_name = 'TSS'

In [5]:
# path to wapiti shared drive
root_path = '/content/drive/Shareddrives/RAPID_Wapiti_project'

In [6]:
data_path = os.path.join(root_path,'water_samples_QC')
csv_files = glob.glob(os.path.join(data_path,f'*_{variable_name}*.csv'))

In [7]:
df = pd.read_csv(csv_files[0])

In [8]:
## Set parameters needed to retrieve streamflow data
start_date1 = '2024-10-01' # Water Year 2025 start

parameter_code = '00060'

end_date2 = '2026-08-07'
stat_id = '00003'

# SF Payette River at Lowman
gages = ['13235000']
gage_names = ['SF Payette River at Lowman']
my_gage = gages[0]

# This section declares the functions that pull and plot data.
Read the doc strings to find out more about the functions.
The function calls are in the next section.

In [9]:
def get_streamflow_data(
    gage: str,
    start_date: str,
    end_date: str,
    parameter_code: str,
    stats_id: str = '00003',
    convert_values: bool = False
) -> pd.DataFrame:
    """Fetches a time series of USGS streamgage observations.

    Requires the `dataretrieval` package maintained by the U.S. Geological Survey.

    Args:
        gage (str): The numeric code for an individual stream gage.
        start_date (str): The starting date of the time series in 'YYYY-MM-DD' format.
        end_date (str): The ending date of the time series in 'YYYY-MM-DD' format.
        parameter_code (str): The parameter code for the measurement of interest
        stats_id (str, optional): The statistical operation resquested. Defaults to '00003' (mean).
            (e.g., '00060' for streamflow).
        convert_values (bool, optional): Option to convert discharge values
            from cfs to cms. Defaults to False.

    Returns:
        pd.DataFrame: A DataFrame containing dates (str) and discharge values (float64)
        in either cfs or cms.
    """

    # 1. Ensure the gage has the required OGC prefix
    location_id = gage if gage.startswith('USGS-') else f'USGS-{gage}'

    # 2. Format the time as an ISO 8601 interval string
    time_interval = f"{start_date}/{end_date}"

    # 3. Call the updated API
    daily_streamflow = waterdata.get_daily(
        monitoring_location_id=location_id,
        parameter_code=parameter_code,
        statistic_id=stats_id,
        time=time_interval
    )
    if stats_id == '00003':
      stat = "daily mean"
    else:
      stat = ""

    print(
        f"Pulled {len(daily_streamflow[0])} {stat} discharge values for gage {gage}"
        )

    dt_output = daily_streamflow[0]['time']


    # Dynamically extract values based on the requested parameter code
    column_name = "value"
    q_output = daily_streamflow[0][column_name].values.astype(float)

    # 4. Accurate conversion factor
    cfs_to_cms = 35.3147

    if convert_values:
        output_data = {
            'date': dt_output,
            'q_cms': q_output / cfs_to_cms
        }
    else:
        output_data = {
            'date': dt_output,
            'q_cfs': q_output
        }

    return pd.DataFrame(output_data)

In [10]:
def merge_with_discharge(sed_df, discharge_df, date_col='date'):
    """
    Merge sediment sample data with discharge time series using the date column.
    Ensures both columns are datetime and aligns on daily dates.
    """
    # Standardize date formats
    sed_df[date_col] = pd.to_datetime(sed_df[date_col], errors='coerce')
    discharge_df[date_col] = pd.to_datetime(discharge_df[date_col], errors='coerce')

    # Drop rows with invalid or missing dates
    sed_df = sed_df.dropna(subset=[date_col])
    discharge_df = discharge_df.dropna(subset=[date_col])

    # Optional: if discharge_df has a datetime index, reset it
    if discharge_df.index.name == date_col:
        discharge_df = discharge_df.reset_index()

    # Merge on the date column
    merged = pd.merge(sed_df, discharge_df, on=date_col, how='left')

    return merged

In [11]:
# extra
def add_season_column(df, date_col='date'):
    """
        Add a sesonal grouping column based on month of sampling
    """
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    month = df[date_col].dt.month

    df['season'] = np.select(
        [
            month.isin([10, 11, 12]),
            month.isin([1, 2, 3]),
            month.isin([4, 5, 6]),
            month.isin([7, 8, 9]),
        ],
        ['OND', 'JFM', 'AMJ', 'JAS'],
        default='Unknown'
    )

    return df

In [12]:
def plot_rating_curve_split_date(df, site_name, split_date, site_col='site',
                                  q_col='q_cfs', y_col='', date_col='date',
                                  figsize=(10, 7), y_ax_label='',
                                  log_y_ax=False, label_before='Before', label_after='After',
                                  save_fig=False,dir_path=None):
    """
    Plot y vs discharge for a SINGLE SITE, with data split into two subsets by date.
    Each subset gets its own regression line and legend.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain site, discharge, y-variable, and date columns.
    site_name : str
        Name of the site to plot.
    split_date : str or datetime
        Date to split the dataset (e.g., '2025-07-01').
    site_col : str
        Column name for site identifier.
    q_col : str
        Column name for discharge values.
    y_col : str
        Column name for y-axis variable (e.g., 'NPOC_conc', 'DIC_conc').
    date_col : str
        Column name for the sampling date.
    figsize : tuple
        Figure size.
    y_ax_label : str
        Label for y-axis.
    log_y_ax : bool
        Whether to use log scale for y-axis.
    label_before : str
        Label for the "before" subset.
    label_after : str
        Label for the "after" subset.
    """

    # Filter data for the site
    site_df = df[df[site_col] == site_name].copy()
    site_df = site_df.dropna(subset=[q_col, y_col, date_col])
    site_df = site_df[(site_df[q_col] > 0) & (site_df[y_col] > 0)]

    if site_df.empty:
        print(f"No valid data for site {site_name}")
        return

    # Convert dates to datetime
    site_df[date_col] = pd.to_datetime(site_df[date_col])
    split_date = pd.to_datetime(split_date)

    # Split into before and after
    df_before = site_df[site_df[date_col] < split_date]
    df_after = site_df[site_df[date_col] >= split_date]

    # Create figure
    fig, ax = plt.subplots(figsize=figsize)

    colors = {'before': 'tab:blue', 'after': 'tab:orange'}

    # Plot before subset
    if not df_before.empty:
        ax.scatter(df_before[q_col], df_before[y_col],
                   color=colors['before'], s=80, edgecolor='k',
                   label=f'{label_before} (n={len(df_before)})', alpha=0.7)

        # Fit regression for before
        if len(df_before) >= 2:
            x = np.log10(df_before[q_col])
            y = np.log10(df_before[y_col])
            slope, intercept, r, p, stderr = linregress(x, y)
            a = 10**intercept
            b = slope

            if np.isfinite(p) and p < 0.05:
                q_fit = np.linspace(df_before[q_col].min(), df_before[q_col].max(), 100)
                ax.plot(q_fit, a * q_fit**b, color=colors['before'], linestyle='-', linewidth=2,
                        label=f'R²={r**2:.2f}, p-value = {p:.3f}')

    # Plot after subset
    if not df_after.empty:
        ax.scatter(df_after[q_col], df_after[y_col],
                   color=colors['after'], s=80, edgecolor='k',
                   label=f'{label_after} (n={len(df_after)})', alpha=0.7)

        # Fit regression for after
        if len(df_after) >= 2:
            x = np.log10(df_after[q_col])
            y = np.log10(df_after[y_col])
            slope, intercept, r, p, stderr = linregress(x, y)
            a = 10**intercept
            b = slope

            if np.isfinite(p) and p < 0.05:
                q_fit = np.linspace(df_after[q_col].min(), df_after[q_col].max(), 100)
                ax.plot(q_fit, a * q_fit**b, color=colors['after'], linestyle='-', linewidth=2,
                        label=f'R²={r**2:.2f}, p-value = {p:.3f}')

    # Log scales
    plt.xscale('log')
    if log_y_ax:
        plt.yscale('log') # could do semilog
        ax.set_ylim(0.1, df[y_col].max())
    else:
        ax.set_ylim(0, df[y_col].max())

    plt.xlabel('Discharge (cfs)',fontsize=8)
    plt.ylabel(y_ax_label, fontsize=8)
    plt.title(f'{site_name}',fontsize=10)
    fig.legend(fontsize=6, loc='outside lower left', ncol=3) # bbox_to_anchor=(0.5, -0.05),
    plt.grid(True, which='both', ls='--', alpha=0.4)
    plt.tight_layout()

    if save_fig:
        plt.savefig(os.path.join(dir_path, f'{site_name}.png'), dpi=300, bbox_inches='tight')
        plt.close()
        print(f'saving {site_name}')
    else:
      plt.show()


# Function calls
This is where the functions run based on the given parameters

In [13]:
sfp_streamflow = get_streamflow_data(
    gage=my_gage,
    start_date=start_date1,
    end_date=end_date2,
    stats_id=stat_id,
    parameter_code=parameter_code
)

sfp_streamflow.describe()

Pulled 674 daily mean discharge values for gage 13235000


,date,q_cfs
count,674,674.000000
mean,2025-09-02 12:08:32.759643904,838.316024
min,2024-10-01 00:00:00,222.000000
25%,2025-03-18 06:00:00,358.000000
50%,2025-09-02 12:00:00,481.000000
75%,2026-02-17 18:00:00,1110.000000
max,2026-08-07 00:00:00,4200.000000
std,NaN,749.917416


In [14]:
merged_df = merge_with_discharge(df, sfp_streamflow)
df_seasonal = add_season_column(merged_df, date_col='date')
site_names = df_seasonal['site'].unique()

In [15]:
split = '2025-11-01' # change this date to vizualize recently added values
# single plots with colored by season
for i in range(0, len(site_names)):
    plot_rating_curve_split_date(
        df_seasonal,
        site_name=site_names[i],
        split_date=split,
        q_col='q_cfs',
        y_col='susp_sed_mg',
        y_ax_label='TSS (mg/L)',
        log_y_ax=True,
        figsize=(3, 3),
        label_before=f'before {split}',
        label_after=f'after {split}',
        save_fig=True,
        dir_path=fig_path
        )

saving Warm_Sp
saving SH_Cr
saving Canyon_Cr_SFP
saving BJ_Cr
saving SFP_Alder_Cr
saving SFP_DW
saving SFP_BJ
saving SFP_Chap
saving Chapman_Cr
saving SFP_Wolf_Cr
saving Fox_Cr
saving SFP_Canyon_Cr
saving Camp_Cr
saving SFP_Wapiti
saving Avalanche_Cr
saving Canyon_cr_avalanche (HP_old)
saving HP
saving GO_Cr
saving Wapiti_Cr
saving Nutcracker_Cr
saving Flat_Cr
saving Bear_Cr
saving MainPayette_Gauge
saving SFP_Staircase
saving MFP_Crouch
saving Canyon_Cr_Banner
No valid data for site Canyon_Cr_DF_1
saving Canyon_Cr_DF_3
saving Canyon_Cr_DF2
saving Canyon_Cr_DF8
saving Canyon_Cr_DF5
saving Canyon_Cr_DF4
saving Canyon_Cr_slump1
saving SFP_Alder_Cr_AM
saving SFP_DW_AM
